# MRCP: validation threshold (non-cropped)

Load the trained model, choose the threshold with the highest validation F1, and apply it to the internal and external test sets. Keep the same images, folder names, and seed used for training.


In [ ]:
%pip install -q torch torchvision scikit-learn numpy matplotlib pillow


In [ ]:
import os
import random
import json

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_curve,
    auc,
    precision_recall_curve,
    classification_report
)
from sklearn.model_selection import StratifiedShuffleSplit

from google.colab import drive


In [ ]:
# -----------------------
# 0. Device & seed
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

PIN_MEMORY = (device.type == "cuda")


In [ ]:
# -----------------------
# 1. Mount Drive & paths
# -----------------------
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/mrcp_data/non_cropped"
INTERNAL_DIR = os.path.join(BASE_DIR, "internal")
EXTERNAL_DIR = os.path.join(BASE_DIR, "external")

OUTPUT_DIR = "/content/drive/MyDrive/mrcp_outputs/non_cropped"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model_path = os.path.join(OUTPUT_DIR, "mrcp_resnet50.pth")
print("Model path:", model_path)

for path in [INTERNAL_DIR, EXTERNAL_DIR]:
    if not os.path.isdir(path):
        raise FileNotFoundError(f"Image directory not found: {path}")
if not os.path.isfile(model_path):
    raise FileNotFoundError(f"Trained model not found: {model_path}")


In [ ]:
# -----------------------
# 2. Transforms & helper Dataset
# -----------------------
IMG_SIZE = 224

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class SubsetWithTransform(Dataset):
    def __init__(self, base_dataset, indices, transform=None):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.transform = transform
        self.samples = [self.base_dataset.samples[i] for i in self.indices]
        self.targets = [self.base_dataset.targets[i] for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = self.base_dataset.loader(path)
        if self.transform is not None:
            img = self.transform(img)
        return img, label


In [ ]:
# -----------------------
# 3. Rebuild datasets & loaders (same split as before)
# -----------------------
internal_base = datasets.ImageFolder(INTERNAL_DIR)
class_names = internal_base.classes
print("Detected classes:", class_names)

if internal_base.class_to_idx != {"0_no_stone": 0, "1_CBD_stone": 1}:
    raise ValueError("Expected class folders: 0_no_stone and 1_CBD_stone.")

num_images = len(internal_base)
all_labels = np.array(internal_base.targets)
all_indices = np.arange(num_images)

# train 70%, val 20%, test 10%, using same random_state=SEED
sss1 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=SEED
)
train_idx, temp_idx = next(sss1.split(all_indices, all_labels))

temp_labels = all_labels[temp_idx]
temp_indices_for_split = np.arange(len(temp_idx))

sss2 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=1/3,  # 1/3 of 30% = 10% total for test
    random_state=SEED
)
val_sub_idx, test_sub_idx = next(sss2.split(temp_indices_for_split, temp_labels))

val_idx = temp_idx[val_sub_idx]
test_idx = temp_idx[test_sub_idx]

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

val_dataset = SubsetWithTransform(internal_base, val_idx, transform=eval_transform)
test_dataset = SubsetWithTransform(internal_base, test_idx, transform=eval_transform)
external_dataset = datasets.ImageFolder(EXTERNAL_DIR, transform=eval_transform)
if external_dataset.class_to_idx != internal_base.class_to_idx:
    raise ValueError("Internal and external class folders must match.")

BATCH_SIZE = 32
NUM_WORKERS = 2

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

external_loader = DataLoader(
    external_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


In [ ]:
# -----------------------
# 4. Rebuild model & load weights
# -----------------------
def build_model():
    model = models.resnet50(weights=None)  # all weights are loaded below
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 1)  # binary
    return model

model = build_model().to(device)

state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
print("Loaded weights from:", model_path)


In [ ]:
# -----------------------
# 5. Find the best F1 threshold on validation
# -----------------------
def evaluate_and_plot(
    model,
    dataloader,
    device,
    dataset_name="Dataset",
    class_names=None,
    save_prefix=None
):
    model.eval()
    all_labels = []
    all_probs = []

    # 1. Collect probabilities
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device).float()

            outputs = model(inputs).view(-1)
            probs = torch.sigmoid(outputs)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    if np.unique(all_labels).size != 2:
        raise ValueError(f"{dataset_name} must contain both classes for ROC evaluation.")

    # 2. PR curve & best F1 threshold
    precision, recall, pr_thresholds = precision_recall_curve(all_labels, all_probs)
    pr_auc = auc(recall, precision)

    f1_scores = []
    for p, r in zip(precision[:-1], recall[:-1]):  # align with thresholds
        if p + r == 0:
            f1_scores.append(0.0)
        else:
            f1_scores.append(2 * p * r / (p + r))
    f1_scores = np.array(f1_scores)

    best_idx = np.argmax(f1_scores)
    best_threshold = pr_thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    # 3. Metrics at best threshold
    preds = (all_probs >= best_threshold).astype(int)

    acc = accuracy_score(all_labels, preds)
    cm = confusion_matrix(all_labels, preds, labels=[0, 1])

    print(f"\n=== {dataset_name} Metrics @ best F1 threshold ===")
    print(f"Best threshold (by F1): {best_threshold:.3f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 (positive class): {best_f1:.4f}")

    print("\nConfusion matrix (rows = true, cols = predicted):")
    print(cm)

    if class_names is not None and len(class_names) == 2:
        print("\nClass order:", class_names)

    print("\nClassification report @ best threshold:")
    print(classification_report(all_labels, preds, labels=[0, 1], digits=3, zero_division=0))

    # 4. Confusion matrix plot
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(f"{dataset_name} - Confusion Matrix @ thr={best_threshold:.2f}")
    plt.colorbar()
    tick_marks = np.arange(2)
    if class_names is None:
        class_names = ["Class 0", "Class 1"]
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    thresh_val = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh_val else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # 5. ROC curve + mark best-threshold point
    fpr, tpr, roc_thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    # Mark the actual predictions at this cutoff.
    tn, fp, fn, tp = cm.ravel()
    best_fpr = fp / (fp + tn)
    best_tpr = tp / (tp + fn)

    plt.figure()
    plt.plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle=':', label="Chance")
    plt.scatter(best_fpr, best_tpr, marker='o', label=f"Best thr={best_threshold:.2f}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate (Recall)")
    plt.title(f"{dataset_name} - ROC Curve (best thr marked)")
    plt.legend(loc="lower right")
    plt.grid(True)
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_roc.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # 6. PR curve with best point marked
    plt.figure()
    plt.plot(recall, precision, label=f"PR (AUC = {pr_auc:.3f})")
    plt.scatter(recall[best_idx], precision[best_idx],
                marker='o',
                label=f"Best F1={best_f1:.2f} @ thr={best_threshold:.2f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{dataset_name} - Precision-Recall Curve")
    plt.legend(loc="best")
    plt.grid(True)
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_pr.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # 7. F1 vs threshold
    plt.figure()
    plt.plot(pr_thresholds, f1_scores, label="F1")
    plt.axvline(best_threshold, linestyle='--',
                label=f"Best thr={best_threshold:.2f}")
    plt.xlabel("Threshold")
    plt.ylabel("F1 score (positive class)")
    plt.title(f"{dataset_name} - F1 vs Threshold")
    plt.legend(loc="best")
    plt.grid(True)
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_f1_threshold.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"\nBest F1 on {dataset_name}: {best_f1:.3f} at threshold {best_threshold:.3f}")

    return {
        "accuracy": float(acc),           # at best_threshold
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "best_f1": float(best_f1),
        "best_threshold": float(best_threshold)
    }


In [ ]:
# -----------------------
# 6. Evaluate with the fixed validation threshold
# -----------------------
def evaluate_and_plot_with_fixed_threshold(
    model,
    dataloader,
    device,
    fixed_threshold,
    dataset_name="Dataset",
    class_names=None,
    save_prefix=None
):
    model.eval()
    all_labels = []
    all_probs = []

    # 1. Collect probabilities
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device).float()

            outputs = model(inputs).view(-1)
            probs = torch.sigmoid(outputs)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    if np.unique(all_labels).size != 2:
        raise ValueError(f"{dataset_name} must contain both classes for ROC evaluation.")

    # 2. Binarise with the fixed threshold
    preds = (all_probs >= fixed_threshold).astype(int)

    cm = confusion_matrix(all_labels, preds, labels=[0, 1])
    acc = accuracy_score(all_labels, preds)

    tn, fp, fn, tp = cm.ravel()
    prec_fixed = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec_fixed = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if prec_fixed + rec_fixed == 0:
        f1_fixed = 0.0
    else:
        f1_fixed = 2 * prec_fixed * rec_fixed / (prec_fixed + rec_fixed)

    print(f"\n=== {dataset_name} Metrics @ fixed threshold {fixed_threshold:.3f} ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec_fixed:.4f}")
    print(f"Recall:    {rec_fixed:.4f}")
    print(f"F1-score:  {f1_fixed:.4f}")

    print("\nConfusion matrix (rows = true, cols = predicted):")
    print(cm)

    if class_names is not None and len(class_names) == 2:
        print("\nClass order:", class_names)

    print("\nClassification report @ fixed threshold:")
    print(classification_report(all_labels, preds, labels=[0, 1], digits=3, zero_division=0))

    # 3. Confusion matrix plot
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(f"{dataset_name} - Confusion Matrix @ thr={fixed_threshold:.2f}")
    plt.colorbar()
    tick_marks = np.arange(2)
    if class_names is None:
        class_names = ["Class 0", "Class 1"]
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)
    thresh_val = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh_val else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # 4. ROC curve (curve itself is threshold-free; mark fixed threshold point)
    fpr, tpr, roc_thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    # Mark the actual predictions at this cutoff.
    tn, fp, fn, tp = cm.ravel()
    best_fpr = fp / (fp + tn)
    best_tpr = tp / (tp + fn)

    plt.figure()
    plt.plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle=':', label="Chance")
    plt.scatter(best_fpr, best_tpr, marker='o',
                label=f"thr={fixed_threshold:.2f}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate (Recall)")
    plt.title(f"{dataset_name} - ROC Curve (fixed thr marked)")
    plt.legend(loc="lower right")
    plt.grid(True)
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_roc.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    # 5. Precision-Recall curve (full curve + fixed thr point)
    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(all_labels, all_probs)
    pr_auc = auc(recall_curve, precision_curve)

    plt.figure()
    plt.plot(recall_curve, precision_curve, label=f"PR (AUC = {pr_auc:.3f})")
    plt.scatter(rec_fixed, prec_fixed,
                marker='o',
                label=f"thr={fixed_threshold:.2f}, F1={f1_fixed:.2f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{dataset_name} - Precision-Recall Curve (fixed thr)")
    plt.legend(loc="best")
    plt.grid(True)
    if save_prefix is not None:
        plt.savefig(f"{save_prefix}_pr.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    return {
        "accuracy": float(acc),
        "precision": float(prec_fixed),
        "recall": float(rec_fixed),
        "f1": float(f1_fixed),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "threshold": float(fixed_threshold),
    }


In [ ]:
# -----------------------
# 7. Select on validation, then evaluate internal and external tests
# -----------------------
print("\nFinding best threshold on VALIDATION set...")
val_metrics = evaluate_and_plot(
    model,
    val_loader,
    device,
    dataset_name="Validation",
    class_names=class_names,
    save_prefix=os.path.join(OUTPUT_DIR, "val_bestthr")
)

best_thr_from_val = val_metrics["best_threshold"]
print("\nBest threshold from validation:", best_thr_from_val)

print("\nEvaluating INTERNAL TEST with validation-based threshold...")
internal_metrics_valthr = evaluate_and_plot_with_fixed_threshold(
    model,
    test_loader,
    device,
    fixed_threshold=best_thr_from_val,
    dataset_name="Internal Test (val thr)",
    class_names=class_names,
    save_prefix=os.path.join(OUTPUT_DIR, "internal_valthr")
)

print("\nEvaluating EXTERNAL TEST with validation-based threshold...")
external_metrics_valthr = evaluate_and_plot_with_fixed_threshold(
    model,
    external_loader,
    device,
    fixed_threshold=best_thr_from_val,
    dataset_name="External Test (val thr)",
    class_names=external_dataset.classes,
    save_prefix=os.path.join(OUTPUT_DIR, "external_valthr")
)

print("\nValidation metrics (best-threshold search):", val_metrics)
print("Internal test metrics @ val thr:", internal_metrics_valthr)
print("External test metrics @ val thr:", external_metrics_valthr)

# ---- Save metrics JSON ----
metrics_valthr = {
    "validation_best": val_metrics,
    "internal_val_threshold": internal_metrics_valthr,
    "external_val_threshold": external_metrics_valthr,
    "best_threshold_from_validation": float(best_thr_from_val),
}
metrics_valthr_path = os.path.join(OUTPUT_DIR, "mrcp_metrics_valthr.json")
with open(metrics_valthr_path, "w") as f:
    json.dump(metrics_valthr, f, indent=4)

print("\nSaved validation-based threshold metrics to:", metrics_valthr_path)
print("All plots and metrics are in:", OUTPUT_DIR)
